In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Week 10 - Neural Networks

Learning contents:
1. Detailed example: Two-layer MLP for regression
    - Forward pass: Calculate the values of $z_1$, $z_2$, and $y$
    - Compute the mean squared error
    - Using backpropagation, compute the gradient or the error w.r.t the weights $w^{(2)}_2$ and $w^{(1)}_{2,2}$ 
    - Compute the updated weights for $w^{(2)}_2$ and $w^{(1)}_{2,2}$ 
2. PyTorch: 2-layer MLP for classification
    - Create and train a 2-layer MLP

# 1) Detailed example: Two-layer MLP for regression
We'll be working through a forward and back-propagation example in all its details for a 2-layer MLP for regression. Our network has the following structure:

![](two-layer-nn.svg)

Where 
    $$
    z_j = \text{ReLU}\left(a_j \right)
    \qquad
    a_j = \sum_i w^{(1)}_{ij} x_i
    \qquad
    y_j = \sum_i w^{(2)}_{ij} z_i
    $$
and the biases 
    $$
    x_0 = z_0 = 1
    $$

Suppose, we have the weights
    $$
    \mathbf{W}^{(1)} = \left[ 
        \begin{matrix}
        0.1 & 0.2\\
        -1.1 & 1.2\\
        -2.1 & 2.2
        \end{matrix}
    \right]
    \qquad
    \mathbf{w}^{(2)} = \left[ 
        \begin{matrix}
        -0.1\\
        1.1 \\
        2.2
        \end{matrix}
    \right]
    $$
Notice that the bias weights are included in the weight matrix. Relating it to the drawing, we have $w_{0,1} = 0.1$, $w_{1,2} = 1.2$

Moreover, we are given an input
    $$
    \mathbf{x} = \left[ 
        \begin{matrix}
        0.1 \\
        0.2
        \end{matrix}
    \right]
    $$
    
Relating these to the drawing, we have $w^{(1)}_{1,2} = 1.2$ and $x_1 = 0.1$

## 1.1) Forward pass: Calculate the values of $z_1$, $z_2$, and $y$

Use the values and formulas given above, as well as information about the ReLU activation function given in lecture 19 sldie 9 to determine the values. 

In [ ]:
x = np.array([1, 0.1, 0.2])
w_1 = np.array([[0.1,  0.2], 
                [-1.1, 1.2], 
                [-2.1, 2.2]])
w_2 = np.array([[-0.1],
                [ 1.1], 
                [ 2.2]])

relu = lambda x: np.maximum(0, x)
ident = lambda x: x
a_1 = w_1.T @ x 
z_1 = np.hstack([1, relu(a_1)]) # Tilføjer bias
a_2 = w_2.T @ z_1
print(f"""
       {z_1[1:]=}
       {y=}
       """)
y = ident(a_2)
print(y)

## 1.2) Compute the mean squared error
Suppose our target $t=2$.
Use the formula for MSE shown in lecture 19, slide 17 (8)

In [ ]:
N = 2
t = 2 
Ew = 1/N * np.sum((y - t)**2)
print(Ew)

## 1.3) Using backpropagation, compute the gradient or the error w.r.t the weights $w^{(2)}_2$ and $w^{(1)}_{2,2}$ 
This corresponds to the bottom row of weights on the figure

To compute this, look at formulas 5.65 - 5.67 in the course book(Bishop).

$$ \delta_k = y_k - t_k $$
$$ \delta_j = h'(a_1) * \sum_{k = 1}^{K} {w_{kj} * \delta_k} $$
$$ \frac{\partial E_n}{\partial w_{ji}^{(1)}} = \delta_j x_i, \quad \quad \frac{\partial E_n}{\partial w_{kj}^{(2)}} = \delta_k z_j$$

$$ f(x) = [\frac{0, \quad x \leq 0}{x \quad\text{else}}], \quad \quad f'(x) = [\frac{0, \quad 1 \leq 0}{x \quad\text{else}}]$$

Jeg ved ikke hvorfor de beregner for (1 - z_j^2) i deltaj. Jeg har en teori om, at det er et simplificeret udtryk af den afledte til logistic sigmoid funktionen. Det passer i hvert fald på det generelle udtryk jeg har. Derfor bruger jeg ikke 5.66, da jeg her bruger en relu funktion

In [ ]:
deltak = y - t
fprime = lambda x: (x > 0).astype(int)
delta_j = fprime(a_1) * (w_2[1:] @ deltak) # w_2[1:] for at fjerne bias-vægten

dw1 = x[:, np.newaxis][1:] @ delta_j[:, np.newaxis].T # Fjerner bias input.
dw2 = z_1[:, np.newaxis].T * deltak
print(dw1)
print(dw2)

## 1.4) Compute the updated weights for $w^{(2)}_2$ and $w^{(1)}_{2,2}$ 
Use a learning rate $\eta = 0.1$ 

You will want to use formula 5.43 in the course book to determine the updated weights
$$ w^{(\tau + 1)} = w^{(\tau)} - \eta \nabla E_n (w^{(\tau)})$$


In [ ]:
eta = 0.1
print(w_1.shape), print(dw1.shape)
print(w_2.shape), print(dw2.shape)
w_1 -= eta * np.vstack([[1, 1], dw1])
w_2 -= eta * dw2.T
print(w_1)
print(w_2)


# 2) PyTorch: 2-layer MLP for classification
Luckily, there are some nice Deep Learning libraries out there, that make working with neural networks a pleasure.
The two most noteable are [Tensorflow](https://www.tensorflow.org) and [PyTorch](https://pytorch.org). We'll be using the latter.

In order to install it in your conda environment you can use
```pip install torch torchvision```

A key feature of these libraries is that they can handle the gradient computation for you.
Moreover, they have a lot of layer types and losses, that are easily composable to handle computation of complex neural networks.

We'll be working with the classic MNIST dataset, which we can easily get via PyTorch

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST('../data', train=True, download=True, transform=transform)
test_data = datasets.MNIST('../data', train=False, download=True, transform=transform)

train_loader = DataLoader(
    train_data,
    batch_size = 64,
    shuffle=True,
)

test_loader = DataLoader(
    test_data,
    batch_size = 64,
)

A few steps happened here:
1. The dataset (train and test) was downloaded 
1. We created a `DataLoader` for each data split. Using this, we get batches of data (64 examples per batch here)
1. We told asked for the training data to be shuffled

Lets see what we get in a batch:

In [ ]:
data, targets = next(iter(train_loader))
data.shape, targets.shape

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure()
for i in range(6):
    plt.subplot(2,3,i+1)
    plt.tight_layout()
    plt.imshow(data[i][0], cmap='gray', interpolation='none')
    plt.title("Ground Truth: {}".format(targets[i]))
    plt.xticks([])
    plt.yticks([])

## 2.1) Create and train a 2-layer MLP
For the network, use a single hidden layer of 512 neurons with a ReLU activation function for the first weight Linear layer. The output of the second Linear layer should be a softmax.

For optimisation, use the SGD optimizer with learning rate of 0.001, and the negative log-likelihood loss.

Train the network for 5 epochs on the train data, and report the prediction accuracy on the test data. You should be able to get about 90% correct.

Hint: check the [PyTorch documentation](https://pytorch.org/docs/stable/index.html) for usage of the layers, optimizers and losses

For this part of the exercise. You will want to replace each "REPLACE" with code correponding to the task described in the line above it. Such as "#Initialise Linear Layers", followed by "REPLACE", where you'd want to replace REPLACE with appropriate code for initialization of linear layers. 

In [ ]:
from torch.nn import Linear, Module
from torch.optim import SGD

import torch.nn.functional as F

class Net(Module):
    def __init__(self):
        super().__init__()
        self.fc1 = Linear(28 * 28, 512)
        self.fc2 = Linear(512, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.log_softmax(self.fc2(x), dim=1)
        return x


def train(model, train_loader, optimizer, epoch, log_interval=50):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()

        if batch_idx % log_interval == 0:
            print(
                f"Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} "
                f"({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}"
            )


def test(model, test_loader):
    model.eval()
    test_loss = 0.0
    correct = 0

    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction="sum").item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)

    print(
        "\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n".format(
            test_loss,
            correct,
            len(test_loader.dataset),
            100.0 * correct / len(test_loader.dataset),
        )
    )

In [ ]:
from torch.nn import Linear, Module
from torch.nn.functional import relu, log_softmax, nll_loss
from torch.optim import SGD

class Net(Module):
    def __init__(self):
        super(Net, self).__init__()
        # Initialise Linear layers
        self.fc1 = Linear(28, 512)          # Input layer to hidden layer
        self.fc2 = Linear(512, 10)          # Hidden layer to output layer


    def forward(self, x):
        # Perform feed-forward computation step
        x = relu(self.fc1(x))
        output = log_softmax(self.fc2(x), dim = 1)
        return output

    
def train(
    model:Module, 
    train_loader:DataLoader, 
    optimizer: SGD, 
    epoch:int, 
    log_interval = 50
):
    # Set model to train mode
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        # Reset the gradients
        optimizer.zero_grad()
        
        # Feed the data through the model
        output = model(data)
        
        # Compute the negative log-likelihood loss
        loss = F.nll_loss(output, target)
        
        # Backward propagate the gradients
        loss.backward()
        
        # Perform an update step using the optimizer
        optimizer.step()

            
        # Log (Optional)
        if batch_idx % log_interval == 0:
            print(
                f"Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)} "
                f"({100. * batch_idx / len(train_loader):.0f}%)]\tLoss: {loss.item():.6f}"
            )
        


def test(model:Module, test_loader:DataLoader):
    # Set model to evaluation mode
    model.eval()
    
    correct = 0
    test_loss = 0.0
    correct = 0
    # Don't accumulate gradients
    with torch.no_grad():
        for data, target in test_loader:
            # Feed the data through the model
            output = model(data)
            
            # Predict the class (it is the index of the max log-probability)
            
            pred = output.argmax(dim=1, keepdim=True)
            
            # <REPLACE>
            
            # Add to the number of correct
            correct += pred.eq(target.view_as(pred)).sum().item()            
    test_loss /= len(test_loader.dataset)

    # Print results
    print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, 
        len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)
    ))
    
    print(
        "\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n".format(
            test_loss,
            correct,
            len(test_loader.dataset),
            100.0 * correct / len(test_loader.dataset),
        )
    )

In [ ]:
model = Net()
optimizer = SGD(model.parameters(), lr=0.001)

epochs = 5
for epoch in range(1, epochs + 1):
    train(model, train_loader, optimizer, epoch, log_interval=50)
    test(model, test_loader)